In [1]:
import pandas as pd

path = "mockdata/"

df_waterings = pd.read_csv(f'{path}waterings.csv')
df_waterings = df_waterings.drop(columns=['PredictedFutureWaterTime'])
df_sensors = pd.read_csv(f'{path}sensor_datas.csv')
df_plants = pd.read_csv(f'{path}plants.csv')

df_sensors

,PlantMAC,Temperature,AirHumidity,SoilHumidity,LightIntensity,Timestamp
0,34:2c:d8:10:0f:2f,NaN,51.24,57.63,1389.79,2025-01-01T12:00:00
1,34:2c:d8:10:0f:2f,20.70,53.24,57.51,1330.55,2025-01-01T14:00:00
2,34:2c:d8:10:0f:2f,18.09,NaN,56.98,NaN,2025-01-01T16:00:00
3,34:2c:d8:10:0f:2f,17.52,51.74,56.71,917.45,2025-01-01T18:00:00
4,34:2c:d8:10:0f:2f,NaN,60.35,56.42,500.00,2025-01-01T20:00:00
...,...,...,...,...,...,...
149995,fb:86:dc:a9:57:9f,24.14,51.79,47.17,913.41,2027-04-14T10:00:00
149996,fb:86:dc:a9:57:9f,21.49,49.09,NaN,972.67,2027-04-14T12:00:00
149997,fb:86:dc:a9:57:9f,22.33,52.44,46.85,1109.57,2027-04-14T14:00:00
149998,fb:86:dc:a9:57:9f,NaN,48.41,47.35,735.54,2027-04-14T16:00:00


In [2]:
#1 MAC Mapping
mac_map = {mac: i for i, mac in enumerate(df_plants['MAC'].unique())}

#2 Prepare waterings

df_waterings = df_waterings.rename(columns={'LastWaterTime': 'WaterTimestamp'})

#3 Convert timestamps
df_sensors['Timestamp']   = pd.to_datetime(df_sensors['Timestamp'])
df_waterings['WaterTimestamp'] = pd.to_datetime(df_waterings['WaterTimestamp'], format='mixed')

#4 Add mac_id to all dataframes
df_sensors['mac_id']   = df_sensors['PlantMAC'].map(mac_map)
df_waterings['mac_id'] = df_waterings['PlantMAC'].map(mac_map)
df_plants['mac_id']    = df_plants['MAC'].map(mac_map)

#5 Clean plants table
df_plants_cleaned = df_plants.drop(columns=['MAC', 'Username', 'Name'])

sensors_sorted = df_sensors.sort_values(['Timestamp']).reset_index(drop=True)
waterings_sorted = df_waterings.sort_values(['WaterTimestamp']).reset_index(drop=True)

sensors_sorted

,PlantMAC,Temperature,AirHumidity,SoilHumidity,LightIntensity,Timestamp,mac_id
0,34:2c:d8:10:0f:2f,NaN,51.24,57.63,1389.79,2025-01-01 12:00:00,0
1,0b:74:8d:a4:a7:7b,21.32,27.85,23.90,8893.63,2025-01-01 12:00:00,12
2,f2:a0:28:84:c9:a4,21.65,44.52,39.63,717.04,2025-01-01 12:00:00,10
3,3a:a1:5f:69:ae:32,19.09,70.87,72.14,1157.03,2025-01-01 12:00:00,5
4,2a:ef:04:8c:db:36,NaN,51.34,51.17,1018.67,2025-01-01 12:00:00,6
...,...,...,...,...,...,...,...
149995,88:04:12:e4:cb:b8,21.23,45.42,NaN,299.05,2027-04-14 18:00:00,1
149996,0b:74:8d:a4:a7:7b,NaN,30.09,9.42,4711.61,2027-04-14 18:00:00,12
149997,34:2c:d8:10:0f:2f,21.44,NaN,44.45,914.28,2027-04-14 18:00:00,0
149998,4d:af:f7:b3:67:dd,19.03,62.38,80.48,1331.09,2027-04-14 18:00:00,9


In [3]:
sensors_sorted.isnull().sum()

PlantMAC              0
Temperature       22898
AirHumidity       22623
SoilHumidity      13811
LightIntensity    22544
Timestamp             0
mac_id                0
dtype: int64

In [4]:
missing_percent = sensors_sorted.isnull().mean() * 100
print(missing_percent)

PlantMAC           0.000000
Temperature       15.265333
AirHumidity       15.082000
SoilHumidity       9.207333
LightIntensity    15.029333
Timestamp          0.000000
mac_id             0.000000
dtype: float64


In [5]:
df_water_sensors = pd.merge_asof(
    sensors_sorted,
    waterings_sorted[['mac_id', 'WaterTimestamp', 'PumpTimeInSeconds', 'WaterLevel']],
    left_on='Timestamp',
    right_on='WaterTimestamp',
    by='mac_id',
    direction='backward',
    tolerance=pd.Timedelta(days=90)
)

df_water_sensors['hours_since_watering'] = (
    df_water_sensors['Timestamp'] - df_water_sensors['WaterTimestamp']
).dt.total_seconds() /3600

# Readings that surpass the plant being watered for 90 days are removed
df_water_sensors = df_water_sensors[~df_water_sensors['hours_since_watering'].isnull()]

df_water_sensors = df_water_sensors.drop(columns=['WaterTimestamp', 'Timestamp'])


# Bring in plant optimal values
df = df_water_sensors.merge(df_plants_cleaned, on='mac_id', how='left')

# Remove impossible outliers
## Team decided the following sensor values are highly like errors above 9000 light,
# above 50C and below 5C, soil humidity above 100 or below 0, air humidity below 5 and above 95

df = df[(df["SoilHumidity"] > 0) & (df["SoilHumidity"]<100) | (df["SoilHumidity"].isnull())]
df = df[(df["Temperature"] > 5) & (df["Temperature"]<50) | (df["Temperature"].isnull())]
df = df[(df["AirHumidity"] > 5) & (df["AirHumidity"]<95) | (df["AirHumidity"].isnull())]
df = df[(df["LightIntensity"] > 0) & (df["LightIntensity"]<9000) | (df["LightIntensity"].isnull())]

df

,PlantMAC,Temperature,AirHumidity,SoilHumidity,LightIntensity,mac_id,PumpTimeInSeconds,WaterLevel,hours_since_watering,OptimalTemperature,OptimalAirHumidity,OptimalSoilHumidity,OptimalLightIntensity
0,34:2c:d8:10:0f:2f,NaN,51.24,57.63,1389.79,0,0.0,88.06,0.0,20.7,50.8,57.5,1282.9
1,0b:74:8d:a4:a7:7b,21.32,27.85,23.90,8893.63,12,0.0,83.09,0.0,21.6,30.9,24.3,6204.3
2,f2:a0:28:84:c9:a4,21.65,44.52,39.63,717.04,10,0.0,93.84,0.0,21.0,39.7,39.9,551.8
3,3a:a1:5f:69:ae:32,19.09,70.87,72.14,1157.03,5,0.0,82.50,0.0,18.8,70.7,72.3,1103.5
4,2a:ef:04:8c:db:36,NaN,51.34,51.17,1018.67,6,0.0,99.11,0.0,21.5,50.2,51.3,716.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...
132913,4f:22:50:8c:d5:46,18.89,48.24,44.05,1192.85,11,151.2,93.75,88.0,19.6,48.6,57.5,1404.9
132914,88:04:12:e4:cb:b8,21.23,45.42,NaN,299.05,1,14.5,6.00,680.0,21.9,45.3,45.4,516.7
132915,0b:74:8d:a4:a7:7b,NaN,30.09,9.42,4711.61,12,9.6,6.00,26.0,21.6,30.9,24.3,6204.3
132916,34:2c:d8:10:0f:2f,21.44,NaN,44.45,914.28,0,35.2,6.00,62.0,20.7,50.8,57.5,1282.9


In [6]:
# Deviation function

df['soil_deficit']= df['OptimalSoilHumidity'] - df['SoilHumidity']
df['soil_deficit_ratio'] = df['soil_deficit'] /df['OptimalSoilHumidity']

df['temp_deviation'] = df['Temperature'] - df['OptimalTemperature']
df['air_hum_deficit'] = df['OptimalAirHumidity'] - df['AirHumidity']
df['light_deviation'] = df['LightIntensity'] - df['OptimalLightIntensity']

# Interaction features
df['deficit_x_temp'] = df['soil_deficit'] * df['temp_deviation']
df['deficit_x_light'] = df['soil_deficit'] * df['light_deviation']
df['deficit_x_air'] = df['soil_deficit'] * df['air_hum_deficit']


# Estimated evaporation
df['et_approx'] = (df['Temperature'] * df['LightIntensity']) / (df['AirHumidity'] + 1)

df = df.drop(columns=['PlantMAC','Temperature','LightIntensity','AirHumidity','SoilHumidity','OptimalAirHumidity','OptimalSoilHumidity','OptimalLightIntensity','OptimalTemperature'])
df

,mac_id,PumpTimeInSeconds,WaterLevel,hours_since_watering,soil_deficit,soil_deficit_ratio,temp_deviation,air_hum_deficit,light_deviation,deficit_x_temp,deficit_x_light,deficit_x_air,et_approx
0,0,0.0,88.06,0.0,-0.13,-0.002261,NaN,-0.44,106.89,NaN,-13.8957,0.0572,NaN
1,12,0.0,83.09,0.0,0.40,0.016461,-0.28,3.05,2689.33,-0.1120,1075.7320,1.2200,6572.346329
2,10,0.0,93.84,0.0,0.27,0.006767,0.65,-4.82,165.24,0.1755,44.6148,-1.3014,341.035062
3,5,0.0,82.50,0.0,0.16,0.002213,0.29,-0.17,53.53,0.0464,8.5648,-0.0272,307.328547
4,6,0.0,99.11,0.0,0.13,0.002534,NaN,-1.14,302.07,NaN,39.2691,-0.1482,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
132913,11,151.2,93.75,88.0,13.45,0.233913,-0.71,0.36,-212.05,-9.5495,-2852.0725,4.8420,457.614470
132914,1,14.5,6.00,680.0,NaN,NaN,-0.67,-0.12,-217.65,NaN,NaN,NaN,136.769313
132915,12,9.6,6.00,26.0,14.88,0.612346,NaN,0.81,-1492.69,NaN,-22211.2272,12.0528,NaN
132916,0,35.2,6.00,62.0,13.05,0.226957,0.74,NaN,-368.62,9.6570,-4810.4910,NaN,NaN


In [7]:
df.isnull().sum()

mac_id                      0
PumpTimeInSeconds           0
WaterLevel                  0
hours_since_watering        0
soil_deficit            10737
soil_deficit_ratio      10737
temp_deviation          17881
air_hum_deficit         17636
light_deviation         19905
deficit_x_temp          26773
deficit_x_light         28517
deficit_x_air           26517
et_approx               46359
dtype: int64

In [8]:
missing_percent = df.isnull().mean() * 100
print(missing_percent)

mac_id                   0.000000
PumpTimeInSeconds        0.000000
WaterLevel               0.000000
hours_since_watering     0.000000
soil_deficit             9.192558
soil_deficit_ratio       9.192558
temp_deviation          15.308944
air_hum_deficit         15.099186
light_deviation         17.041806
deficit_x_temp          22.921893
deficit_x_light         24.415031
deficit_x_air           22.702717
et_approx               39.690585
dtype: float64
